
# POC — can 'Unhealthy' trees be seen **statistically** in SAR mean-backscatter?

**Standalone diagnostic notebook.

## Motivation / goal
The main notebook's per-tree ML model barely beats no-skill under *honest* spatial-block CV
(PR-AUC ~0.15 vs ~0.09 baseline; prior POC reports). The leading hypothesis is that **disease
spreads regionally** — an Unhealthy tree almost never sits alone; a regional patch of trees turns
Unhealthy together.

This POC therefore asks a simpler, **statistical (non-ML)** question:

> If we sample SAR backscatter (dB) at every tree coordinate with a 3x3 mean window and a
> 5x5 mean window, is the Unhealthy group distinguishable from the Healthy group (by central
> tendency, spread, or distribution)? And does *widening* the window (3->5->7->...->21 px)
> reveal or widen the gap (the "regional spread" hypothesis)?

## Pipeline
1. Pulls the **GS bucket** (`gs://sar-oilpalm`).
2. Load the **label CSVs** (tree `id / Long / Lat / Class`) and the **dB intensity GeoTIFF**.
3. Sample the backscatter **mean** at each tree with 3x3 and 5x5 windows (plus a few larger
   windows for the regional-spread probe).
4. Run descriptive + inferential statistics (mean / median / std / Cohen's d / Mann-Whitney U)
   and write CSV / JSON / PNG artifacts under the output `poc_mean_sampling/` folder.


In [24]:

# Install the one extra package (rasterio) that a fresh Colab kernel may lack.
# scipy / seaborn / matplotlib / pandas ship with Colab by default.
!pip install -q rasterio
import rasterio
print("rasterio", rasterio.__version__)


rasterio 1.5.1



## 1. Setup, auth & GCS config
Aligned with the main notebook's constants (`GCS_BUCKET`, `SCENE_NAME`, `PROJECT_ROOT`).


In [25]:

# ============================ setup / config ==============================
import os, glob, json, sys, warnings, subprocess, shutil
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
from scipy.stats import mannwhitneyu

from google.colab import auth
auth.authenticate_user()
print("Google auth: OK")

# ---------------- paths & config (mirrors the main notebook) ----------------
GCS_BUCKET  = "sar-oilpalm"               # <-- your bucket name
SCENE_NAME  = "ALOS2-HBQR1_1__D-ORBIT__ALOS2650583560-260610_Cal_ML_Spk_TC"
PROJECT_ROOT = "/content/SAR-OilPalm"

DATA_DIR    = os.path.join(PROJECT_ROOT, "data")
SCENE_DIR   = os.path.join(DATA_DIR, "SAR-scenes")
OUTPUT_DIR  = os.path.join(DATA_DIR, "processed")
RESULTS_DIR = os.path.join(OUTPUT_DIR, "poc_mean_sampling")
PLOTS_DIR   = os.path.join(RESULTS_DIR, "plots")
LABEL_DIR   = os.path.join(DATA_DIR, "Label-status")

for _d in (SCENE_DIR, OUTPUT_DIR, RESULTS_DIR, PLOTS_DIR, LABEL_DIR):
    os.makedirs(_d, exist_ok=True)

INTENSITY_TIF = os.path.join(SCENE_DIR, SCENE_NAME + ".tif")
LOCATIONS     = ["Palong_Basemap", "Serting_Basemap"]

# Sampling windows (odd ints). 3x3 and 5x5 are the headline pair; the larger
# windows test the hypothesis that disease affects a whole REGION of trees.
WINDOW_TREND = [3, 5, 7, 9, 11, 15, 21]
HEADLINE_W   = [3, 5]

# Candidate GCS zip names of the '<loc> Classification.csv' label CSVs.
# Edit if your bucket layout uses different names.
LABEL_ZIPS = {
    "Palong_Basemap":  ["Label_Palong2022.zip", "Label_Palong2021.zip"],
    "Serting_Basemap": ["Label_Serting2022.zip"],
}

print("Project root :", PROJECT_ROOT)
print("Intensity TIF:", INTENSITY_TIF)
print("Locations    :", LOCATIONS)
print("Artifacts    :", RESULTS_DIR)


Google auth: OK
Project root : /content/SAR-OilPalm
Intensity TIF: /content/SAR-OilPalm/data/SAR-scenes/ALOS2-HBQR1_1__D-ORBIT__ALOS2650583560-260610_Cal_ML_Spk_TC.tif
Locations    : ['Palong_Basemap', 'Serting_Basemap']
Artifacts    : /content/SAR-OilPalm/data/processed/poc_mean_sampling



## 2. Pull the data from GCS
Downloads the intensity GeoTIFF + label CSV zip(s) into a local cache (the same
source the main notebook uses). `gcloud storage` is available on every Colab runtime.


In [26]:

# ============================ GCS read-in ================================
def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print("  ...! command failed:", " ".join(cmd), "\n", r.stderr[-500:])
    return r.returncode == 0

def gcs_cp(remote, local, tag=""):
    if os.path.exists(local):
        print("[cached]", tag or os.path.basename(local))
        return True
    print("[gcs] ", tag or os.path.basename(local))
    return _run(["gcloud", "storage", "cp", remote, local])

# ---- 1) intensity dB GeoTIFF ----
_remote_tif = f"gs://{GCS_BUCKET}/data/SAR-scenes/{SCENE_NAME}.tif"
if not gcs_cp(_remote_tif, INTENSITY_TIF, "intensity dB GeoTIFF"):
    if not os.path.exists(INTENSITY_TIF):
        raise FileNotFoundError("intensity GeoTIFF download failed - check GCS_BUCKET / SCENE_NAME / IAM scopes")

# ---- 2) label (tree coordinate) CSVs ----
def locate_label_csv(loc):
    hits = sorted(glob.glob(os.path.join(LABEL_DIR, "**", f"{loc} Classification.csv"), recursive=True))
    if hits:
        return hits[0]
    for zf in LABEL_ZIPS.get(loc, []):
        local = os.path.join(LABEL_DIR, zf)
        if not gcs_cp(f"gs://{GCS_BUCKET}/data/Label-status/{zf}", local, zf):
            continue
        try:
            shutil.unpack_archive(local, LABEL_DIR)
        except Exception as e:
            print("  unpack failed:", e)
        if os.path.exists(local):
            os.remove(local)
        hits = sorted(glob.glob(os.path.join(LABEL_DIR, "**", f"{loc} Classification.csv"), recursive=True))
        if hits:
            return hits[0]
    return None

LABEL_CSVS = {}
for _loc in LOCATIONS:
    _p = locate_label_csv(_loc)
    if _p is None:
        raise FileNotFoundError(
            f"label CSV for '{_loc}' not found. Put '<{_loc}> Classification.csv' "
            f"under {LABEL_DIR} or fix LABEL_ZIPS."
        )
    LABEL_CSVS[_loc] = _p
    print("  ", _loc, "=", _p)

print("[data ready]")


[cached] intensity dB GeoTIFF
   Palong_Basemap = /content/SAR-OilPalm/data/Label-status/Result/Palong_Basemap Classification.csv
   Serting_Basemap = /content/SAR-OilPalm/data/Label-status/Result/Serting_Basemap Classification.csv
[data ready]



## 3. Inspect the intensity raster & build the multi-window mean sampler
CRS is **EPSG:4326** (WGS84) in the labels; we reproject into the TIFF's CRS and
read a centered window per tree. To keep I/O cheap we read the largest window once
and *crop* it to each size (3x3, 5x5, ...) around the tree pixel.


In [27]:

import rasterio
from rasterio.warp import transform as _transform
from rasterio.windows import Window

with rasterio.open(INTENSITY_TIF) as _s:
    _meta = dict(crs=_s.crs, grid=(_s.width, _s.height), res=_s.res,
                 nodata=_s.nodata, bands=_s.count)
print("Raster metadata:", _meta)

# Intensity product band order (HH, HV, VV, VH).  Adjust only if inspection shows
# a different ordering in YOUR file (see main.ipynb band-resolution logic).
band_labels = ["HH", "HV", "VV", "VH"]

def sample_windows(tif, lon, lat, windows):
    """For each tree coordinate, read a max-sized window once then crop to each
    window size. Returns {win: (n, nbands)} of *mean dB* (NaN-masked)."""
    max_win = max(windows)
    half    = max_win // 2
    xs, ys  = _transform("EPSG:4326", tif.crs, lon, lat)
    n       = len(lon)
    out     = {w: np.full((n, tif.count), np.nan) for w in windows}
    for i in range(n):
        row, col = (int(v) for v in tif.index(xs[i], ys[i]))
        w0 = Window(col - half, row - half, max_win, max_win)
        w0 = w0.intersection(Window(0, 0, tif.width, tif.height))
        if w0.width < 1 or w0.height < 1:
            continue
        data = tif.read(window=w0).astype(float)
        if tif.nodata is not None:
            data[data == tif.nodata] = np.nan
        data[data <= -99.0] = np.nan
        lo_r, lo_c = row - w0.row_off, col - w0.col_off
        for w in windows:
            hw = w // 2
            r0, c0 = lo_r - hw, lo_c - hw
            if r0 < 0 or c0 < 0 or r0 + w > w0.height or c0 + w > w0.width:
                continue
            d = data[:, r0:r0 + w, c0:c0 + w]
            for b in range(tif.count):
                v = d[b][np.isfinite(d[b])]
                if v.size:
                    out[w][i, b] = float(np.mean(v))
    return out


Raster metadata: {'crs': CRS.from_wkt('PROJCS["WGS 84 / UTM zone 48N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",105],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32648"]]'), 'grid': (8437, 11862), 'res': (6.4301092, 6.4301092), 'nodata': None, 'bands': 4}



## 4. Read the labels, sample at every tree coordinate
Produces the wide artifact `backscatter_means.csv` (one row per tree: `id, Long,
Lat, Class, Location` + ``<pol>_meanW<n>`` for each band and window size).


In [28]:

wide_frames = []
with rasterio.open(INTENSITY_TIF) as src:
    for loc in LOCATIONS:
        ldf = pd.read_csv(LABEL_CSVS[loc])
        ldf.columns = [str(c).strip() for c in ldf.columns]
        if "ID" in ldf.columns:
            ldf = ldf.rename(columns={"ID": "id"})
        ldf["Class"]  = ldf["Class"].astype(str).str.strip()
        ldf["Health"] = ldf["Class"]
        ldf["Location"] = loc

        res = sample_windows(src, ldf["Long"].tolist(), ldf["Lat"].tolist(),
                             WINDOW_TREND)
        for w in WINDOW_TREND:
            for b, pol in enumerate(band_labels):
                ldf[f"{pol}_meanW{w}"] = res[w][:, b]
        wide_frames.append(ldf)

wide = pd.concat(wide_frames, ignore_index=True)
wide_path = os.path.join(RESULTS_DIR, "backscatter_means.csv")
wide.to_csv(wide_path, index=False)
print("wide shape :", wide.shape)
print("class counts (all locations):")
print(wide.groupby(["Location", "Class"]).size().to_string())
print("saved ->", wide_path)


wide shape : (4709, 34)
class counts (all locations):
Location         Class    
Palong_Basemap   Healthy      1971
                 Middle        493
                 Unhealthy     200
Serting_Basemap  Healthy      1513
                 Middle        378
                 Unhealthy     154
saved -> /content/SAR-OilPalm/data/processed/poc_mean_sampling/backscatter_means.csv



## 5. Statistical analysis (non-ML)

**Health classes kept:** `Healthy` (negative) vs `Unhealthy` (positive). Rows labelled
`Middle` are kept in the wide CSV but excluded here to keep the two-class contrast clean.

For every (location, band, window) we compute:
- descriptive stats per class (n, mean, std, median),
- **Cohen's d** (pooled SD effect size),
- a **Mann-Whitney U / Wilcoxon** rank test p-value (two groups vs separation),
- `frac_unhealthy_below_healthy_median` — % of Unhealthy samples at/below the Healthy
  median. `>0.5` means Healthier trees are systematically higher, `<0.5` the reverse;
  this is a simple, robust stand-in for rank-separation.


In [29]:

HEALTHY, UNHEALTHY = "Healthy", "Unhealthy"

def stats(h, u):
    """Descriptive + inferential stats between two numeric series."""
    h = pd.to_numeric(h, errors="coerce").dropna()
    u = pd.to_numeric(u, errors="coerce").dropna()
    if len(h) < 2 or len(u) < 2:
        return None
    mh, mu = float(h.mean()), float(u.mean())
    sh, su = float(h.std(ddof=0)), float(u.std(ddof=0))
    medh, medu = float(h.median()), float(u.median())
    sd_pool = float(np.sqrt((float(h.var(ddof=1)) + float(u.var(ddof=1))) / 2.0))
    cohen = (mu - mh) / sd_pool if sd_pool > 0 else float("nan")
    try:
        pval = mannwhitneyu(h, u, method="asymptotic").pvalue
        pval = float(pval)
    except Exception:
        pval = float("nan")
    frac_below = float((u <= medh).mean())
    return {
        "n_h": int(len(h)), "n_u": int(len(u)),
        "mean_h": round(mh, 4), "mean_u": round(mu, 4),
        "std_h": round(sh, 4), "std_u": round(su, 4),
        "median_h": round(medh, 4), "median_u": round(medu, 4),
        "diff": round(mu - mh, 4),
        "cohen_d": round(cohen, 4),
        "mwu_p": pval,
        "frac_unhealthy_below_healthy_median": round(frac_below, 4),
    }

# ---------- headline: Healthy vs Unhealthy at 3x3 and 5x5 ----------
rows = []
for loc in LOCATIONS:
    sub = wide[wide["Location"] == loc]
    for w in HEADLINE_W:
        for pol in band_labels:
            col = f"{pol}_meanW{w}"
            s = sub[[col, "Class"]].dropna()
            h = s.loc[s["Class"] == HEALTHY, col]
            u = s.loc[s["Class"] == UNHEALTHY, col]
            r = stats(h, u)
            if r is None:
                continue
            r.update({"location": loc, "band": pol, "window": w})
            rows.append(r)
stat_tab = pd.DataFrame(rows).sort_values(["location", "window", "band"])
order = ["location", "band", "window", "n_h", "n_u", "mean_h", "mean_u", "std_h",
         "std_u", "median_h", "median_u", "diff", "cohen_d", "mwu_p",
         "frac_unhealthy_below_healthy_median"]
stat_tab = stat_tab[[c for c in order if c in stat_tab.columns]]
stat_tab.to_csv(os.path.join(RESULTS_DIR, "stats_h_vs_u.csv"), index=False)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
print(stat_tab.to_string(index=False))


       location band  window  n_h  n_u  mean_h  mean_u  std_h  std_u  median_h  median_u    diff  cohen_d    mwu_p  frac_unhealthy_below_healthy_median
 Palong_Basemap   HH       3 1971  200  0.1534  0.1386 0.0684 0.0453    0.1398    0.1342 -0.0148  -0.2556 0.014531                               0.5450
 Palong_Basemap   HV       3 1971  200  0.0227  0.0211 0.0080 0.0051    0.0209    0.0208 -0.0016  -0.2428 0.161502                               0.5350
 Palong_Basemap   VH       3 1971  200  0.1099  0.0905 0.0712 0.0317    0.0895    0.0843 -0.0195  -0.3526 0.000517                               0.5450
 Palong_Basemap   VV       3 1971  200  0.0229  0.0212 0.0077 0.0053    0.0214    0.0206 -0.0017  -0.2552 0.034493                               0.5750
 Palong_Basemap   HH       5 1971  200  0.1523  0.1405 0.0604 0.0413    0.1396    0.1376 -0.0118  -0.2275 0.076216                               0.5200
 Palong_Basemap   HV       5 1971  200  0.0226  0.0211 0.0074 0.0042    0.0207    0.0206


## 6. Regional-delta: is the *change* from 3x3 to 5x5 different by class?
If disease hits a *region*, enlarging the window from 3x3 to 5x5 slides the mean
towards whatever surrounds the tree. This cell tests whether that per-tree delta
separates Healthy vs Unhealthy better than the 3x3 / 5x5 means do.


In [30]:

delta_rows = []
for loc in LOCATIONS:
    sub = wide[wide["Location"] == loc]
    for pol in band_labels:
        d5 = pd.to_numeric(sub[f"{pol}_meanW5"], errors="coerce")
        d3 = pd.to_numeric(sub[f"{pol}_meanW3"], errors="coerce")
        tmp = pd.DataFrame({"Class": sub["Class"].values, "delta": (d5 - d3).values})
        tmp = tmp.dropna()
        h = tmp.loc[tmp["Class"] == HEALTHY, "delta"]
        u = tmp.loc[tmp["Class"] == UNHEALTHY, "delta"]
        r = stats(h, u)
        if r is None:
            continue
        r.update({"location": loc, "band": pol})
        delta_rows.append(r)
delta_tab = pd.DataFrame(delta_rows).sort_values(["location", "band"])
cols = ["location", "band"] + [c for c in order if c in delta_tab.columns and c not in ("window",)]
delta_tab = delta_tab[[c for c in cols if c in delta_tab.columns]]
delta_tab.to_csv(os.path.join(RESULTS_DIR, "delta_stats.csv"), index=False)
print(delta_tab.to_string(index=False))


       location band        location band  n_h  n_u  mean_h  mean_u  std_h  std_u  median_h  median_u    diff  cohen_d    mwu_p  frac_unhealthy_below_healthy_median
 Palong_Basemap   HH  Palong_Basemap   HH 1971  200 -0.0012  0.0019 0.0199 0.0119    0.0009    0.0026  0.0031   0.1863 0.004601                               0.4250
 Palong_Basemap   HV  Palong_Basemap   HV 1971  200 -0.0001  0.0001 0.0019 0.0017    0.0001    0.0003  0.0002   0.1072 0.049526                               0.4150
 Palong_Basemap   VH  Palong_Basemap   VH 1971  200 -0.0008  0.0011 0.0233 0.0088    0.0002    0.0019  0.0020   0.1125 0.010350                               0.4300
 Palong_Basemap   VV  Palong_Basemap   VV 1971  200 -0.0001  0.0000 0.0020 0.0015    0.0000    0.0002  0.0001   0.0658 0.391510                               0.4500
Serting_Basemap   HH Serting_Basemap   HH 1513  154 -0.0000  0.0008 0.0135 0.0129    0.0011    0.0024  0.0008   0.0623 0.456939                               0.4481
Serting_Ba


## 7. Window-size trend — does widening the mean reveal the regional signal?
For each (location, band) we sweep the window 3 -> 21 px and record mean-H, mean-U,
the difference and the Mann-Whitney p-value. If Unhealthy trees sit inside a regional
patch of altered backscatter, |difference| / significance should improve with window.


In [31]:

trend_rows = []
for loc in LOCATIONS:
    sub = wide[wide["Location"] == loc]
    for w in WINDOW_TREND:
        for pol in band_labels:
            col = f"{pol}_meanW{w}"
            s = sub[[col, "Class"]].dropna()
            h = s.loc[s["Class"] == HEALTHY, col]
            u = s.loc[s["Class"] == UNHEALTHY, col]
            r = stats(h, u)
            if r is None:
                continue
            trend_rows.append({"location": loc, "band": pol, "window": int(w),
                               "mean_h": r["mean_h"], "mean_u": r["mean_u"],
                               "diff": r["diff"], "cohen_d": r["cohen_d"],
                               "mwu_p": r["mwu_p"]})
trend_tab = pd.DataFrame(trend_rows).sort_values(["location", "band", "window"])
trend_tab.to_csv(os.path.join(RESULTS_DIR, "window_trend.csv"), index=False)
print(trend_tab.pivot_table(index=["band"], columns=["window"],
                            values="cohen_d", aggfunc="mean").round(3).to_string())


window     3      5      7      9      11     15     21
band                                                   
HH     -0.154 -0.131 -0.088 -0.057 -0.032  0.020  0.038
HV     -0.032  0.002  0.012  0.018  0.029  0.032 -0.001
VH     -0.144 -0.164 -0.179 -0.172 -0.165 -0.126 -0.137
VV      0.006  0.005 -0.010 -0.023 -0.035 -0.054 -0.094



## 8. Plots
Figures are written to `poc_mean_sampling/plots/`. They visualise the same quantities
as the tables, plus a spatial map so we can see whether 'Unhealthy' is spatially
contiguous (consistent with a *regional* agent).

The backscatter-density (KDE) figure is generated for **both** headline windows:
`kde_3x3_by_class.png` and `kde_5x5_by_class.png`.

Panels sharing a band column are drawn on a **common scale** (same dB x-range and same density y-range) across locations and window sizes, so they can be compared directly.


In [32]:

os.makedirs(PLOTS_DIR, exist_ok=True)

def save(fig, name):
    p = os.path.join(PLOTS_DIR, name)
    fig.savefig(p, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("  saved", p)

paletteC = {"Healthy": "#2e8b57", "Unhealthy": "#c0392b"}

# ---- Fig 1: boxplots per (loc, band) for windows 3 & 5 ----
# NB: palette here maps the `hue` categories (the two window sizes), not the
# x categories (Health classes). `hue_order` pins "3x3" / "5x5" ordering.
paletteW = {"3x3": "#3c78a8", "5x5": "#d1835c"}
fig, axes = plt.subplots(len(LOCATIONS), len(band_labels),
                         figsize=(4 * len(band_labels), 3.2 * len(LOCATIONS)), squeeze=False)
for r, loc in enumerate(LOCATIONS):
    sub = wide[wide["Location"] == loc]
    for c, pol in enumerate(band_labels):
        ax = axes[r][c]
        d = sub[["Class", f"{pol}_meanW3", f"{pol}_meanW5"]].dropna()
        # Build from numpy arrays -> no index alignment / duplicate-label issues
        vals3 = d[f"{pol}_meanW3"].to_numpy(dtype=float)
        vals5 = d[f"{pol}_meanW5"].to_numpy(dtype=float)
        classes = d["Class"].to_numpy()
        dd = pd.DataFrame({
            "Class": np.concatenate([classes, classes]),
            "dB":    np.concatenate([vals3, vals5]),
            "win":   ["3x3"] * len(d) + ["5x5"] * len(d),
        })
        sns.boxplot(data=dd, x="Class", y="dB", hue="win", ax=ax,
                    palette=paletteW, order=[HEALTHY, UNHEALTHY],
                    hue_order=["3x3", "5x5"], linewidth=0.8)
        ax.set_title(f"{loc} - {pol}")
        ax.set_xlabel("")
        ax.legend(title="", fontsize=7, loc="best")
fig.suptitle("Mean backscatter by Health class (3x3 vs 5x5)", y=1.00)
save(fig, "boxplot_3x3_vs_5x5.png")

# ---- Fig 2: KDE overlap per (loc, band) -- one figure PER HEADLINE WINDOW ----
# Produces kde_3x3_by_class.png AND kde_5x5_by_class.png (the "Backscatter Density" plots).
# SCALE IS MADE CONSISTENT PER BAND COLUMN: identical x (dB) and y (density) limits are
# applied to every panel of a band across BOTH locations and BOTH window sizes, so the
# panels can be compared directly. Limits come from the pooled samples:
#   x: 0.5-99.5th percentile (+10% pad) -- keeps outlier spikes from flattening every panel;
#   y: max fitted-KDE height over all panels of that band (+8% headroom).
from scipy.stats import gaussian_kde

_kde_axes = {}   # (_W, loc, pol) -> ax
_kde_vals = {}   # (_W, loc, pol) -> {cls: Series}
for _W in HEADLINE_W:
    fig, axes = plt.subplots(len(LOCATIONS), len(band_labels),
                             figsize=(4.5 * len(band_labels), 3.2 * len(LOCATIONS)), squeeze=False)
    for r, loc in enumerate(LOCATIONS):
        sub = wide[wide["Location"] == loc]
        for c, pol in enumerate(band_labels):
            ax = axes[r][c]
            d = sub[[f"{pol}_meanW{_W}", "Class"]].dropna()
            for cls, color in paletteC.items():
                vals = d.loc[d["Class"] == cls, f"{pol}_meanW{_W}"].astype(float)
                if len(vals.unique()) > 1:
                    sns.kdeplot(vals, ax=ax, label=cls, color=color, fill=True, alpha=0.3)
                    _kde_vals.setdefault((_W, loc, pol), {})[cls] = vals
            ax.set_title(f"{loc} - {pol} ({_W}x{_W})")
            ax.set_xlabel("dB")
            _kde_axes[(_W, loc, pol)] = ax
    fig.suptitle(f"Backscatter density, Healthy vs Unhealthy @ {_W}x{_W}", y=1.00)

# --- enforce the per-band common scale across panels / windows / locations ---
for c, pol in enumerate(band_labels):
    pool = pd.concat([v for key, vv in _kde_vals.items() if key[2] == pol
                      for v in vv.values()])
    if pool.empty:
        continue
    lo_q, hi_q = pool.quantile([0.005, 0.995])
    pad = 0.10 * (hi_q - lo_q)
    x_lo, x_hi = float(lo_q - pad), float(hi_q + pad)
    grid = np.linspace(x_lo, x_hi, 512)
    dens_max = 0.0
    for (_W, _loc, _pol), vv in _kde_vals.items():
        if _pol != pol:
            continue
        for vals in vv.values():
            if vals.nunique() < 2:
                continue
            dens_max = max(dens_max, float(gaussian_kde(vals)(grid).max()))
    y_hi = dens_max * 1.08
    for (_W, _loc, _pol), ax in _kde_axes.items():
        if _pol != pol:
            continue
        ax.set_xlim(x_lo, x_hi)
        ax.set_ylim(0, y_hi)

for _W in HEADLINE_W:
    _fig = _kde_axes[(_W, LOCATIONS[0], band_labels[0])].figure
    _fig.axes[0].legend()
    save(_fig, f"kde_{_W}x{_W}_by_class.png")

# ---- Fig 3: cohen's d vs window (regional-spread probe) ----
fig, axes = plt.subplots(1, len(LOCATIONS), figsize=(7 * len(LOCATIONS), 4.5), squeeze=False)
for lid, loc in enumerate(LOCATIONS):
    ax = axes[0][lid]
    tt = trend_tab[trend_tab["location"] == loc]
    for pol in band_labels:
        tp = tt[tt["band"] == pol]
        ax.plot(tp["window"], tp["cohen_d"], marker="o", label=pol)
    ax.axhline(0, color="k", lw=0.6)
    ax.axhline(0.2, color="gray", ls="--", lw=0.6)
    ax.axhline(-0.2, color="gray", ls="--", lw=0.6)
    ax.set_title(f"{loc} - effect size vs window")
    ax.set_xlabel("window size (px)"); ax.set_ylabel("Cohen's d (U - H)")
    ax.legend()
fig.suptitle("Regional-spread probe: does the class gap widen with window?", y=1.02)
save(fig, "window_trend_effect.png")

# ---- Fig 4: spatial maps (class + dB) ----
fig, axes = plt.subplots(1, len(LOCATIONS), figsize=(6.5 * len(LOCATIONS), 5.5), squeeze=False)
for lid, loc in enumerate(LOCATIONS):
    ax = axes[0][lid]
    sub = wide[wide["Location"] == loc]
    for cls, color in paletteC.items():
        d = sub[sub["Class"] == cls]
        ax.scatter(d["Long"], d["Lat"], c=color, s=2, alpha=0.6, label=cls)
    ax.set_title(loc)
    ax.set_xlabel("Long"); ax.set_ylabel("Lat"); ax.legend(markerscale=4)
fig.suptitle("Tree locations by Health class", y=1.02)
save(fig, "spatial_scan.png")

print("plots dir:", PLOTS_DIR)


  saved /content/SAR-OilPalm/data/processed/poc_mean_sampling/plots/boxplot_3x3_vs_5x5.png
  saved /content/SAR-OilPalm/data/processed/poc_mean_sampling/plots/kde_5x5_by_class.png
  saved /content/SAR-OilPalm/data/processed/poc_mean_sampling/plots/window_trend_effect.png
  saved /content/SAR-OilPalm/data/processed/poc_mean_sampling/plots/spatial_scan.png
plots dir: /content/SAR-OilPalm/data/processed/poc_mean_sampling/plots



## 9. Compact JSON summary (for the analysis pass)
Writes `poc_summary.json` with the headline numbers: best discriminative band per
location, and the *pooled* (both locations) Healthy-vs-Unhealthy effect at 3x3 & 5x5.


In [33]:

summary = {
    "bucket": GCS_BUCKET,
    "scene": SCENE_NAME,
    "locations": LOCATIONS,
    "windows_sampled": WINDOW_TREND,
    "headline_windows": HEADLINE_W,
}

# ---- per location: best (band,window) discriminator ----
summary["per_location"] = {}
for loc in LOCATIONS:
    tt = trend_tab[trend_tab["location"] == loc].copy()
    tt["absd"] = tt["cohen_d"].abs()
    best = tt.sort_values("absd", ascending=False).iloc[0]
    summary["per_location"][loc] = {
        "best_band": str(best["band"]),
        "best_window": int(best["window"]),
        "best_cohens_d": round(float(best["cohen_d"]), 4),
        "best_mannwhitney_p": best["mwu_p"],
        "n_significant_band_window_pairs": int((tt["mwu_p"] < 0.05).sum()),
    }

# ---- pooled across locations, at the two headline windows ----
summary["pooled_headline"] = []
for w in HEADLINE_W:
    entry = {"window": int(w)}
    for pol in band_labels:
        col = f"{pol}_meanW{w}"
        s = wide[[col, "Class"]].dropna()
        h = s.loc[s["Class"] == HEALTHY, col]
        u = s.loc[s["Class"] == UNHEALTHY, col]
        r = stats(h, u)
        if r is None:
            continue
        entry[pol] = {
            "cohens_d": r["cohen_d"], "mwu_p": r["mwu_p"],
            "frac_unhealthy_below_healthy_median": r["frac_unhealthy_below_healthy_median"],
            "n_h": r["n_h"], "n_u": r["n_u"],
        }
    summary["pooled_headline"].append(entry)

with open(os.path.join(RESULTS_DIR, "poc_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))


{
  "bucket": "sar-oilpalm",
  "scene": "ALOS2-HBQR1_1__D-ORBIT__ALOS2650583560-260610_Cal_ML_Spk_TC",
  "locations": [
    "Palong_Basemap",
    "Serting_Basemap"
  ],
  "windows_sampled": [
    3,
    5,
    7,
    9,
    11,
    15,
    21
  ],
  "headline_windows": [
    3,
    5
  ],
  "per_location": {
    "Palong_Basemap": {
      "best_band": "VH",
      "best_window": 5,
      "best_cohens_d": -0.356,
      "best_mannwhitney_p": 0.0035082495598302275,
      "n_significant_band_window_pairs": 5
    },
    "Serting_Basemap": {
      "best_band": "VV",
      "best_window": 5,
      "best_cohens_d": 0.274,
      "best_mannwhitney_p": 0.001172867280133847,
      "n_significant_band_window_pairs": 11
    }
  },
  "pooled_headline": [
    {
      "window": 3,
      "HH": {
        "cohens_d": -0.1845,
        "mwu_p": 0.01577161863611991,
        "frac_unhealthy_below_healthy_median": 0.5734,
        "n_h": 3484,
        "n_u": 354
      },
      "HV": {
        "cohens_d": -0.0977,
